# Fake News Detection - Model Training Pipeline

Welcome to the **Fake News Detection** end-to-end model training notebook using the **WELFake** dataset (72,000+ news articles).

### Pipeline Steps:
1. **Environment Setup & Imports**: Import data science, NLP, and machine learning libraries.
2. **Dataset Loading**: Ingest `Dataset.csv` from `../data/`.
3. **Missing Value Treatment**: Clean null values across `title` and `text`, concatenating them into a unified feature.
4. **NLTK Text Preprocessing**: Lowercasing, punctuation & URL removal, stopword removal, and cached lemmatization.
5. **Feature Extraction**: Fit a TF-IDF vectorizer (unigrams + bigrams).
6. **Model Training & Comparison**:
   - **Logistic Regression**
   - **PassiveAggressiveClassifier**
7. **Evaluation**: Compute Accuracy, Confusion Matrices, and Classification Reports.
8. **Artifact Persistence**: Save the winning model (`fake_news_model.pkl`) and vectorizer (`tfidf_vectorizer.pkl`) to `../models/` for downstream FastAPI integration.

## 1. Imports & Configuration

In [ ]:
import os
import re
import sys
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# NLP - NLTK
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# Plotting config
%matplotlib inline
sns.set_theme(style="whitegrid")
print("[+] Libraries loaded successfully!")

## 2. Paths & Directory Setup

In [ ]:
BASE_DIR = os.path.dirname(os.path.abspath("")) if "__file__" not in locals() else os.getcwd()
DATA_PATH = os.path.join(BASE_DIR, "..", "data", "Dataset.csv")
MODELS_DIR = os.path.join(BASE_DIR, "..", "models")

os.makedirs(MODELS_DIR, exist_ok=True)
print(f"Dataset location : {os.path.abspath(DATA_PATH)}")
print(f"Models directory : {os.path.abspath(MODELS_DIR)}")

## 3. Dataset Ingestion & Missing Value Cleaning

In [ ]:
# Load dataset
df = pd.read_csv(DATA_PATH)
print(f"Initial dataset shape: {df.shape}")
display(df.head())

# Missing value analysis
print("\n--- Missing Values Before Cleaning ---")
print(df.isnull().sum())

# Handle missing values by replacing NaNs in title and text with empty strings
df['title'] = df['title'].fillna('')
df['text'] = df['text'].fillna('')

# Combine title and text into a unified 'content' feature
df['content'] = df['title'] + ' ' + df['text']

# Ensure valid labels and non-empty content
df = df[df['label'].notnull()]
df['label'] = df['label'].astype(int)
df = df[df['content'].str.strip() != '']

print("\n--- Cleaned Dataset ---")
print(f"Cleaned dataset shape: {df.shape}")
print(f"Target distribution:\n{df['label'].value_counts()}")

## 4. NLP Preprocessing Pipeline
Applies text lowercasing, removal of URLs, punctuation and special symbols, stopword removal, and lemmatization (with memoization cache for high throughput).

In [ ]:
# Initialize NLTK components
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
lemma_cache = {}

url_pattern = re.compile(r'https?://\S+|www\.\S+|<.*?>')
clean_pattern = re.compile(r'[^a-z\s]')

def fast_lemmatize(w):
    if w not in lemma_cache:
        lemma_cache[w] = lemmatizer.lemmatize(w)
    return lemma_cache[w]

def preprocess_text(text: str) -> str:
    if not isinstance(text, str) or not text:
        return ''
    text = url_pattern.sub(' ', text.lower())
    text = clean_pattern.sub(' ', text)
    words = text.split()
    tokens = [fast_lemmatize(w) for w in words if w not in stop_words and len(w) > 2]
    return ' '.join(tokens)

# Test preprocessor on a sample
sample = "Breaking News! The president announced new economic measures today at https://news.example.com."
print("Sample raw:     ", sample)
print("Sample cleaned: ", preprocess_text(sample))

In [ ]:
# Apply preprocessing to entire dataset
print("[*] Preprocessing text across all articles...")
t0 = time.time()
df['cleaned_content'] = [preprocess_text(t) for t in df['content']]
print(f"[+] Preprocessing finished in {time.time() - t0:.2f} seconds.")
print(f"Vocabulary cache size: {len(lemma_cache):,} unique lemmas.")

## 5. Train-Test Split & TF-IDF Vectorization

In [ ]:
# Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_content'],
    df['label'],
    test_size=0.20,
    random_state=42,
    stratify=df['label']
)

print(f"Train samples: {len(X_train):,} | Test samples: {len(X_test):,}")

# Fit TF-IDF Vectorizer
print("[*] Vectorizing text data with TF-IDF...")
tfidf_vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"[+] TF-IDF Matrix shape: {X_train_tfidf.shape}")

## 6. Model Training & Evaluation
We train and compare two top-performing linear models for high-dimensional sparse NLP classification:
1. **Logistic Regression**
2. **PassiveAggressiveClassifier**

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Passive Aggressive Classifier": PassiveAggressiveClassifier(max_iter=1000, random_state=42, C=0.5)
}

results = {}

for name, model in models.items():
    print("=" * 60)
    print(f"[*] Training {name}...")
    t_start = time.time()
    model.fit(X_train_tfidf, y_train)
    train_time = time.time() - t_start
    
    # Inference
    y_pred = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=["Real (0)", "Fake (1)"])
    
    results[name] = {
        "model": model,
        "accuracy": acc,
        "confusion_matrix": cm,
        "report": report,
        "train_time": train_time
    }
    
    print(f"[+] {name} trained in {train_time:.2f}s")
    print(f"[+] Accuracy: {acc * 100:.2f}%\n")
    print("Confusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(report)

## 7. Visualizing Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (name, res) in enumerate(results.items()):
    disp = ConfusionMatrixDisplay(confusion_matrix=res["confusion_matrix"], display_labels=["Real (0)", "Fake (1)"])
    disp.plot(ax=axes[idx], cmap="Blues", colorbar=False)
    axes[idx].set_title(f"{name}\nAccuracy: {res['accuracy'] * 100:.2f}%")
    axes[idx].grid(False)

plt.tight_layout()
plt.show()

## 8. Exporting Winning Model & Vectorizer

In [ ]:
# Determine champion model
best_name = max(results, key=lambda k: results[k]['accuracy'])
best_model = results[best_name]['model']
best_acc = results[best_name]['accuracy']

print(f"[★] Champion Model: {best_name} with {best_acc * 100:.2f}% accuracy")

model_path = os.path.join(MODELS_DIR, "fake_news_model.pkl")
vectorizer_path = os.path.join(MODELS_DIR, "tfidf_vectorizer.pkl")

print(f"[*] Saving model to: {model_path}")
joblib.dump(best_model, model_path)

print(f"[*] Saving vectorizer to: {vectorizer_path}")
joblib.dump(tfidf_vectorizer, vectorizer_path)

print("[✔] All artifacts saved successfully and ready for FastAPI deployment!")